# 5: TF-IDF (Term Frequency-Inverse Document Frequency)
**Goal:** Use TF-IDF to score words in each abstract and extract the most important ones as a generated title.

---


## How TF-IDF Works

TF-IDF works by scoring each word in a document by two factors:

- **TF (Term Frequency):** How often does this word appear in *this* abstract? Words that appear more are likely more relevant to the paper.
- **IDF (Inverse Document Frequency):** How rare is this word across *all* abstracts? Words that appear in every paper (like "the", "we", "study") carry little meaning. Words that are rare across the corpus but frequent in this abstract are likely the most distinctive and important.

$$\text{TF-IDF}(w, d) = \text{TF}(w, d) \times \log\frac{N}{df(w)}$$

Where $N$ is the total number of documents and $df(w)$ is how many documents contain word $w$.


**Why this is better than the bigram baseline:**  
Unlike the bigram model, TF-IDF actually reads the abstract. It identifies words that are specific and important to *this* paper rather than just generating the most statistically common title words. The output will vary per abstract.

**Implementation:** Thankfully I can use `TfidfVectorizer` from `scikit-learn` out-of-the-box.

## Imports

In [4]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab", quiet=True)

True

## Loading Data

In [5]:
df = pd.read_parquet("../ext/sample_50k_preprocessed.parquet")

train_df, temp_df = train_test_split(df, test_size=0.2,  random_state=42)
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train : {len(train_df):,}")
print(f"Test  : {len(test_df):,}")

Train : 40,000
Test  : 5,000


## Cleaning Text

In [6]:
LATEX_PATTERNS = [
    (r"\$[^$]*\$", ""),
    (r"\\[a-zA-Z]+\{[^}]*\}", ""),
    (r"\\[a-zA-Z]+", ""),
    (r"\{[^}]*\}", ""),
    (r"\s+", " "),
]

def clean_text(text):
    if not isinstance(text, str):
        return ""
    for pattern, replacement in LATEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text.encode("ascii", errors="ignore").decode().strip().lower()

for split in [train_df, test_df]:
    split["abstract_clean"] = split["abstract"].apply(clean_text)
    split["title_clean"]    = split["title"].apply(clean_text)

## Fitting Vector
We fit the vectorizer on training abstracts only. This will learn the IDF weights (how rare or common each word is across the corpus). We then use these weights to score words in any new abstract.

In [9]:
vectorizer = TfidfVectorizer(
    max_features=20000, #keep top 20k words by frequency
    stop_words="english", #remove common stopwords
    ngram_range=(1, 1),  #unigrams only
)

vectorizer.fit(train_df["abstract_clean"])

print(f"Vocab size: {len(vectorizer.vocabulary_):,}")
print(f"Sample vocab: {list(vectorizer.vocabulary_.keys())[:10]}")

Vocab size: 20,000
Sample vocab: ['employing', 'lattice', 'theory', 'majorization', 'investigate', 'universal', 'quantum', 'uncertainty', 'relation', 'number']


vectorizer.fit() -> will fit on training abstracts only - no data leakage

## Generating Titles

Idea is to score each word in the abstract using TF-IDF weights. Then to return the highest scoring words as the title. The word ordering is preserved from the original abstract.

In [12]:
def generate_title_tfidf(abstract_text, top_n=8):
    tfidf_vector = vectorizer.transform([abstract_text]) #transforms the abstract into a TF-IDF vector

    feature_names = vectorizer.get_feature_names_out() #get feature names and their scores for this abstract
    scores        = tfidf_vector.toarray()[0]

    #pair each word with its score and sort by score descending:
    word_scores = [(feature_names[i], scores[i]) for i in scores.argsort()[::-1] if scores[i] > 0]

    #take top_n words
    top_words = set(w for w, _ in word_scores[:top_n])

    #reconstruct in original abstract word order for readability
    ordered = [w for w in abstract_text.split() if w in top_words]

    #deduplicate (while preserving order)
    seen, result = set(), []
    for w in ordered:
        if w not in seen:
            seen.add(w)
            result.append(w)

    return " ".join(result)


print("Sample outputs:\n")
for i in range(10):
    row = test_df.iloc[i]
    print(f"[{i+1}]")
    print(f"  Abstract  : {row['abstract_clean'][:120]}...")
    print(f"  Reference : {row['title']}")
    print(f"  Generated : {generate_title_tfidf(row['abstract_clean'])}")
    print()

Sample outputs:

[1]
  Abstract  : we present a superconvergent hybridizable discontinuous galerkin (hdg) method for the steady-state incompressible navier...
  Reference : A superconvergent HDG method for the Incompressible Navier-Stokes
  Equations on general polyhedral meshes
  Generated : polyhedral velocity trace superconvergence postprocessing

[2]
  Abstract  : we study the asymptotic dynamics of stochastic young differential delay equations under the regular assumptions on lipsc...
  Reference : Pullback attractors for stochastic Young differential delay equations
  Generated : delay lipschitz continuity random possesses pullback attractor parts

[3]
  Abstract  : the process of hand washing, according to the who, is divided into stages with clearly defined two handed dynamic gestur...
  Reference : Tracking Hand Hygiene Gestures with Leap Motion Controller
  Generated : hand stages features gesture leap hands palm gestures

[4]
  Abstract  : new corrections to the energy of -l

## 5.6 Observations

Compared to the bigram baseline, TF-IDF shows an improvement in one distinct way: the output changes per abstract. Each generated title reflects words that are actually distinctive to that specific paper rather than the same generic sequence every time.

However, it still has notable limitations:

- Every word in the generated title comes directly from the abstract — it cannot rephrase or synthesize
- It scores words independently with no awareness of how they relate to each other grammatically
- A word that appears once but is highly domain-specific may score high even if it is not the most title-worthy term

Better then our baseline, but it'll be ideal to find something that addresses some of these limitation (BART or T5).